In [1]:
!which python

/opt/miniconda3/envs/mappings/bin/python


In [2]:
%load_ext autoreload
%autoreload 2
from IPython.core.interactiveshell import InteractiveShell

In [3]:
# basic packages
import os
import re
import sys
import datetime
from typing import List, Dict, Tuple, Optional, Any
from itertools import combinations, product
from pathlib import Path
import glob
#import yaml
import tqdm
import multiprocessing as mp

In [4]:
# data science
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [5]:
# bioinformatics
import pandas as pd
from Bio.Seq import MutableSeq
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment
from bintools.utils.utils import get_yaml_config

In [6]:
ROOT_dir = Path(os.path.abspath(os.path.join(Path("../")))).__str__()
if ROOT_dir not in sys.path:
    sys.path.append(ROOT_dir)

In [7]:
list_of_geneID_simu: List[str] = get_yaml_config(ROOT_dir+"/configs/configs-CpG.yaml")["simulation"]["geneID"]
list_of_geneID_emp: List[str] = get_yaml_config(ROOT_dir+"/configs/configs-CpG.yaml")["empirical"]["geneID"]

In [8]:
def sign_95(x,):
     return np.sum(x >= 0.95) / x.shape[0] * 100
    
def sign_99(x,):
     return np.sum(x >= 0.99) / x.shape[0] * 100

def sign_90(x,):
     return np.sum(x >= 0.90) / x.shape[0] * 100
     
def prop(x):
     return np.sum(x) / x.shape[0]

def tran(x):
    if x <= 1:
        return 0
    else:
        return 1


def concat(input_dir:str, pattern:str):
     files: List[str] = glob.glob(input_dir + pattern)
     assert len(files) > 0
     list_of_df : List[pd.DataFrame] = []
     for f in files:
          cur_df: pd.DataFrame = pd.read_csv(f,sep="\t")
          list_of_df += [cur_df]
     return pd.concat(list_of_df,axis=0,ignore_index=True)

def recover_data_emp(input_dir, pattern)-> List[pd.DataFrame]:
        
    set_completed: set = set()
    list_of_df: List[pd.DataFrame] = []
    list_of_files: List[str] = glob.glob(input_dir + pattern)
    for f in list_of_files:
        GENEID = f.split("/")[-1].split("-")[0]
        df: pd.DataFrame = pd.read_csv(f, sep="\t")
        df["geneID"] = [GENEID]*df.shape[0]
        if df.shape[0] == 1000:
            list_of_df += [df]
            set_completed.add(GENEID)
        else:
            print(f"Error in {GENEID}")
    print( set(list_of_geneID_emp) - set_completed)
    return pd.concat(list_of_df,ignore_index=True)


def recover_data_sim(input_dir, pattern, expected_n_lines)-> List[pd.DataFrame]:
    list_of_df: List[pd.DataFrame] = []
    list_of_files: List[str] = glob.glob(input_dir + pattern)
    for f in list_of_files:
        GENEID = f.split("/")[-1].split("-")[0]
        OMEGA = float(f.split("/")[-1].split("-")[2])
        CPG = float(f.split("/")[-1].split("-")[3])
        TPA = float(f.split("/")[-1].split("-")[4])
        TBL = float(f.split("/")[-1].split("-")[5])
        DRAWID = int(f.split("/")[-1].split("-")[6])
        df: pd.DataFrame = pd.read_csv(f, sep="\t")
        df["geneID"] = [GENEID]*df.shape[0]
        df["omega"] = [OMEGA]*df.shape[0]
        df["CpG"] = [CPG]*df.shape[0]
        df["TpA"] = [TPA]*df.shape[0]
        df["tbl"] = [TBL]*df.shape[0]
        df["drawID"] = [DRAWID]*df.shape[0]
        if df.shape[0] == expected_n_lines:
            list_of_df += [df]
    return pd.concat(list_of_df,ignore_index=True)

## Mappings


### Empirical


#### MG-F1x4W

In [9]:
input_dir = f"{ROOT_dir}/outputs/empirical/pbmpi/MG-F1x4W/"
pattern = "*-A.TsCpGRate"
df_concat_m0gtr = recover_data_emp(input_dir=input_dir,pattern=pattern)

set()


In [10]:
set(list_of_geneID_emp) - set(df_concat_m0gtr["geneID"].unique())  #list_of_geneID_emp

set()

In [11]:
assert 1000 * 137 == df_concat_m0gtr.shape[0]

In [12]:
df_concat_m0gtr.groupby(["geneID","mcmcID","type"]).agg(["count"])

CG CG>CA CG>TG  CG12 CG>CA12 CG>TG12  CG23 CG>CA23  \
                   count count count count   count   count count   count   
geneID mcmcID type                                                         
ACO1   100    post    10    10    10    10      10      10    10      10   
              pred    10    10    10    10      10      10    10      10   
       102    post    10    10    10    10      10      10    10      10   
              pred    10    10    10    10      10      10    10      10   
       104    post    10    10    10    10      10      10    10      10   
...                  ...   ...   ...   ...     ...     ...   ...     ...   
ZBTB24 194    pred    10    10    10    10      10      10    10      10   
       196    post    10    10    10    10      10      10    10      10   
              pred    10    10    10    10      10      10    10      10   
       198    post    10    10    10    10      10      10    10      10   
              pred    10    10    10    10      10      10    10      10   

                   CG>TG23  CG31  ... CG>TGsyn12 CG>TGnonsyn12 CG>CAsyn23  \
                     count count  ...      count         count      count   
geneID mcmcID type                ...                                       
ACO1   100    post      10    10  ...         10            10         10   
              pred      10    10  ...         10            10         10   
       102    post      10    10  ...         10            10         10   
              pred      10    10  ...         10            10         10   
       104    post      10    10  ...         10            10         10   
...                    ...   ...  ...        ...           ...        ...   
ZBTB24 194    pred      10    10  ...         10            10         10   
       196    post      10    10  ...         10            10         10   
              pred      10    10  ...         10            10         10   
       198    post      10    10  ...         10            10         10   
              pred      10    10  ...         10            10         10   

                   CG>CAnonsyn23 CG>TGsyn23 CG>TGnonsyn23 CG>CAsyn31  \
                           count      count         count      count   
geneID mcmcID type                                                     
ACO1   100    post            10         10            10         10   
              pred            10         10            10         10   
       102    post            10         10            10         10   
              pred            10         10            10         10   
       104    post            10         10            10         10   
...                          ...        ...           ...        ...   
ZBTB24 194    pred            10         10            10         10   
       196    post            10         10            10         10   
              pred            10         10            10         10   
       198    post            10         10            10         10   
              pred            10         10            10         10   

                   CG>CAnonsyn31 CG>TGsyn31 CG>TGnonsyn31  
                           count      count         count  
geneID mcmcID type                                         
ACO1   100    post            10         10            10  
              pred            10         10            10  
       102    post            10         10            10  
              pred            10         10            10  
       104    post            10         10            10  
...                          ...        ...           ...  
ZBTB24 194    pred            10         10            10  
       196    post            10         10            10  
              pred            10         10            10  
       198    post            10         10            10  
              pred            10         10            10  

[13700 rows x 28 columns]

In [13]:
df_concat_m0gtr["CpGRate"] = (df_concat_m0gtr["CG>TG"]+df_concat_m0gtr["CG>CA"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRatesyn"] = (df_concat_m0gtr["CG>TGsyn"]+df_concat_m0gtr["CG>CAsyn"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRatenonsyn"] = (df_concat_m0gtr["CG>TGnonsyn"]+df_concat_m0gtr["CG>CAnonsyn"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRate12"] = (df_concat_m0gtr["CG>TG12"]+df_concat_m0gtr["CG>CA12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRate23"] = (df_concat_m0gtr["CG>TG23"]+df_concat_m0gtr["CG>CA23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRate31"] = (df_concat_m0gtr["CG>TG31"]+df_concat_m0gtr["CG>CA31"])/df_concat_m0gtr["CG31"]

df_concat_m0gtr["CpGRatesyn12"] = (df_concat_m0gtr["CG>TGsyn12"]+df_concat_m0gtr["CG>CAsyn12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRatesyn23"] = (df_concat_m0gtr["CG>TGsyn23"]+df_concat_m0gtr["CG>CAsyn23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRatesyn31"] = (df_concat_m0gtr["CG>TGsyn31"]+df_concat_m0gtr["CG>CAsyn31"])/df_concat_m0gtr["CG31"]

df_concat_m0gtr["CpGRatenonsyn12"] = (df_concat_m0gtr["CG>TGnonsyn12"]+df_concat_m0gtr["CG>CAnonsyn12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRatenonsyn23"] = (df_concat_m0gtr["CG>TGnonsyn23"]+df_concat_m0gtr["CG>CAnonsyn23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRatenonsyn31"] = (df_concat_m0gtr["CG>TGnonsyn31"]+df_concat_m0gtr["CG>CAnonsyn31"])/df_concat_m0gtr["CG31"]


df_concat_m0gtr.groupby(["geneID","type"]).agg([np.mean,np.std])\
    [[
        "CpGRate","CG>TG","CG>CA","CG",\
        "CpGRatesyn","CG>TGsyn","CG>CAsyn",\
        "CpGRatenonsyn","CG>TGnonsyn","CG>CAnonsyn",\

        "CpGRate12","CG>TG12", "CG>CA12", "CG12",\
        "CpGRate23","CG>TG23", "CG>CA23", "CG23",\
        "CpGRate31","CG>TG31", "CG>CA31", "CG31",\
    
        "CpGRatesyn12","CG>TGsyn12", "CG>CAsyn12", \
        "CpGRatesyn23","CG>TGsyn23", "CG>CAsyn23", \
        "CpGRatesyn31","CG>TGsyn31", "CG>CAsyn31", \
    
        "CpGRatenonsyn12","CG>TGnonsyn12", "CG>CAnonsyn12", \
        "CpGRatenonsyn23","CG>TGnonsyn23", "CG>CAnonsyn23", \
        "CpGRatenonsyn31","CG>TGnonsyn31", "CG>CAnonsyn31", \
        
    ]].round(2)

CpGRate         CG>TG          CG>CA              CG          \
               mean   std    mean    std    mean    std     mean     std   
geneID type                                                                
ACO1   post    0.43  0.02  203.31  12.58  136.25   7.99   794.95   21.66   
       pred    0.21  0.02  249.19  26.83  276.25  29.29  2537.43  170.78   
ACSL3  post    0.32  0.02  109.77   8.60   76.89   6.93   580.07   21.61   
       pred    0.22  0.02  196.82  24.66  192.57  23.90  1792.42  144.13   
ADNP   post    0.32  0.02  129.18  10.23   95.08   8.23   699.41   23.15   
...             ...   ...     ...    ...     ...    ...      ...     ...   
WDR91  pred    0.16  0.01  309.39  29.38  309.29  27.29  3821.49  223.44   
YTHDF2 post    0.32  0.03   31.26   2.86    7.46   2.60   122.83    5.79   
       pred    0.25  0.03   57.95  10.88   61.31  11.57   484.54   54.30   
ZBTB24 post    0.41  0.02  142.71   9.26  170.92  11.03   773.61   23.63   
       pred    0.19  0.01  200.60  23.39  293.76  29.12  2634.24  172.46   

            CpGRatesyn        ... CG>TGnonsyn23       CG>CAnonsyn23       \
                  mean   std  ...          mean   std          mean  std   
geneID type                   ...                                          
ACO1   post       0.33  0.02  ...          8.90  1.68           0.0  0.0   
       pred       0.18  0.01  ...         17.17  4.46           0.0  0.0   
ACSL3  post       0.26  0.02  ...          1.98  1.31           0.0  0.0   
       pred       0.19  0.02  ...         13.93  3.95           0.0  0.0   
ADNP   post       0.28  0.02  ...          4.51  1.35           0.0  0.0   
...                ...   ...  ...           ...   ...           ...  ...   
WDR91  pred       0.14  0.01  ...         21.51  5.06           0.0  0.0   
YTHDF2 post       0.30  0.03  ...          1.11  0.72           0.0  0.0   
       pred       0.22  0.03  ...          3.03  1.82           0.0  0.0   
ZBTB24 post       0.30  0.02  ...         13.89  2.56           0.0  0.0   
       pred       0.15  0.01  ...         22.21  5.21           0.0  0.0   

            CpGRatenonsyn31       CG>TGnonsyn31      CG>CAnonsyn31        
                       mean   std          mean  std          mean   std  
geneID type                                                               
ACO1   post            0.26  0.03           0.0  0.0         36.46  3.16  
       pred            0.07  0.02           0.0  0.0         20.38  5.08  
ACSL3  post            0.15  0.03           0.0  0.0         16.08  2.60  
       pred            0.06  0.02           0.0  0.0         14.72  4.13  
ADNP   post            0.11  0.02           0.0  0.0         20.31  3.07  
...                     ...   ...           ...  ...           ...   ...  
WDR91  pred            0.05  0.01           0.0  0.0         22.23  4.90  
YTHDF2 post            0.05  0.05           0.0  0.0          0.58  0.60  
       pred            0.10  0.06           0.0  0.0          3.27  1.85  
ZBTB24 post            0.30  0.03           0.0  0.0         49.88  4.12  
       pred            0.09  0.02           0.0  0.0         33.66  6.88  

[274 rows x 80 columns]

In [14]:
df_concat_m0gtr.groupby(["geneID","type"]).agg([np.mean,])[[
        "CpGRate","CG>TG","CG>CA","CG",\
        "CpGRatesyn","CG>TGsyn","CG>CAsyn",\
        "CpGRatenonsyn","CG>TGnonsyn","CG>CAnonsyn",\

        "CpGRate12","CG>TG12", "CG>CA12", "CG12",\
        "CpGRate23","CG>TG23", "CG>CA23", "CG23",\
        "CpGRate31","CG>TG31", "CG>CA31", "CG31",\
    
        "CpGRatesyn12","CG>TGsyn12", "CG>CAsyn12", \
        "CpGRatesyn23","CG>TGsyn23", "CG>CAsyn23", \
        "CpGRatesyn31","CG>TGsyn31", "CG>CAsyn31", \
    
        "CpGRatenonsyn12","CG>TGnonsyn12", "CG>CAnonsyn12", \
        "CpGRatenonsyn23","CG>TGnonsyn23", "CG>CAnonsyn23", \
        "CpGRatenonsyn31","CG>TGnonsyn31", "CG>CAnonsyn31", \
        
    ]].round(2)

CpGRate   CG>TG   CG>CA       CG CpGRatesyn CG>TGsyn CG>CAsyn  \
               mean    mean    mean     mean       mean     mean     mean   
geneID type                                                                 
ACO1   post    0.43  203.31  136.25   794.95       0.33   189.04    72.88   
       pred    0.21  249.19  276.25  2537.43       0.18   218.61   236.05   
ACSL3  post    0.32  109.77   76.89   580.07       0.26    96.59    51.99   
       pred    0.22  196.82  192.57  1792.42       0.19   172.49   163.71   
ADNP   post    0.32  129.18   95.08   699.41       0.28   123.36    70.12   
...             ...     ...     ...      ...        ...      ...      ...   
WDR91  pred    0.16  309.39  309.29  3821.49       0.14   269.76   265.22   
YTHDF2 post    0.32   31.26    7.46   122.83       0.30    30.05     6.75   
       pred    0.25   57.95   61.31   484.54       0.22    52.67    54.72   
ZBTB24 post    0.41  142.71  170.92   773.61       0.30   124.24   110.17   
       pred    0.19  200.60  293.76  2634.24       0.15   161.05   227.27   

            CpGRatenonsyn CG>TGnonsyn CG>CAnonsyn  ... CG>CAsyn31  \
                     mean        mean        mean  ...       mean   
geneID type                                        ...              
ACO1   post          0.10       14.26       63.37  ...        0.0   
       pred          0.03       30.57       40.19  ...        0.0   
ACSL3  post          0.07       13.18       24.90  ...        0.0   
       pred          0.03       24.33       28.87  ...        0.0   
ADNP   post          0.04        5.82       24.96  ...        0.0   
...                   ...         ...         ...  ...        ...   
WDR91  pred          0.02       39.63       44.07  ...        0.0   
YTHDF2 post          0.02        1.20        0.71  ...        0.0   
       pred          0.02        5.28        6.59  ...        0.0   
ZBTB24 post          0.10       18.47       60.75  ...        0.0   
       pred          0.04       39.54       66.49  ...        0.0   

            CpGRatenonsyn12 CG>TGnonsyn12 CG>CAnonsyn12 CpGRatenonsyn23  \
                       mean          mean          mean            mean   
geneID type                                                               
ACO1   post            0.07          5.37         26.91            0.05   
       pred            0.03         13.40         19.81            0.02   
ACSL3  post            0.08         11.21          8.82            0.01   
       pred            0.03         10.40         14.15            0.02   
ADNP   post            0.02          1.31          4.64            0.02   
...                     ...           ...           ...             ...   
WDR91  pred            0.02         18.12         21.84            0.01   
YTHDF2 post            0.00          0.09          0.12            0.04   
       pred            0.02          2.25          3.32            0.01   
ZBTB24 post            0.05          4.58         10.87            0.05   
       pred            0.04         17.33         32.83            0.02   

            CG>TGnonsyn23 CG>CAnonsyn23 CpGRatenonsyn31 CG>TGnonsyn31  \
                     mean          mean            mean          mean   
geneID type                                                             
ACO1   post          8.90           0.0            0.26           0.0   
       pred         17.17           0.0            0.07           0.0   
ACSL3  post          1.98           0.0            0.15           0.0   
       pred         13.93           0.0            0.06           0.0   
ADNP   post          4.51           0.0            0.11           0.0   
...                   ...           ...             ...           ...   
WDR91  pred         21.51           0.0            0.05           0.0   
YTHDF2 post          1.11           0.0            0.05           0.0   
       pred          3.03           0.0            0.10           0.0   
ZBTB24 post         13.89           0.0

In [15]:
dict_of_stats = {}
k = 0
rowiter = iter(df_concat_m0gtr.iterrows())
while ((row_post := next(rowiter, None)) is not None):
    row_pred = next(rowiter)
    dict_of_stats[k] = {
        "CpGRate_post": row_post[1]["CpGRate"],
        "CpGRate_pred": row_pred[1]["CpGRate"],
        "CpGRate_comp": row_post[1]["CpGRate"]>row_pred[1]["CpGRate"],

        "CpGRatesyn_post": row_post[1]["CpGRatesyn"],
        "CpGRatesyn_pred": row_pred[1]["CpGRatesyn"],
        "CpGRatesyn_comp": row_post[1]["CpGRatesyn"]>row_pred[1]["CpGRatesyn"],

        "CpGRatenonsyn_post": row_post[1]["CpGRatenonsyn"],
        "CpGRatenonsyn_pred": row_pred[1]["CpGRatenonsyn"],
        "CpGRatenonsyn_comp": row_post[1]["CpGRatenonsyn"]>row_pred[1]["CpGRatenonsyn"],

        "CpGRate12_post": row_post[1]["CpGRate12"],
        "CpGRate12_pred": row_pred[1]["CpGRate12"],
        "CpGRate12_comp": row_post[1]["CpGRate12"]>row_pred[1]["CpGRate12"],
        "CpGRate23_post": row_post[1]["CpGRate23"],
        "CpGRate23_pred": row_pred[1]["CpGRate23"],
        "CpGRate23_comp": row_post[1]["CpGRate23"]>row_pred[1]["CpGRate23"],
        "CpGRate31_post": row_post[1]["CpGRate31"],
        "CpGRate31_pred": row_pred[1]["CpGRate31"],
        "CpGRate31_comp": row_post[1]["CpGRate31"]>row_pred[1]["CpGRate31"],

        "CpGRatesyn12_post": row_post[1]["CpGRatesyn12"],
        "CpGRatesyn12_pred": row_pred[1]["CpGRatesyn12"],
        "CpGRatesyn12_comp": row_post[1]["CpGRatesyn12"]>row_pred[1]["CpGRatesyn12"],
        "CpGRatesyn23_post": row_post[1]["CpGRatesyn23"],
        "CpGRatesyn23_pred": row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn23_comp": row_post[1]["CpGRatesyn23"]>row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn31_post": row_post[1]["CpGRatesyn31"],
        "CpGRatesyn31_pred": row_pred[1]["CpGRatesyn31"],
        "CpGRatesyn31_comp": row_post[1]["CpGRatesyn31"]>row_pred[1]["CpGRatesyn31"],


        "CpGRatenonsyn12_post": row_post[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn12_pred": row_pred[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn12_comp": row_post[1]["CpGRatenonsyn12"]>row_pred[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn23_post": row_post[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_pred": row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_comp": row_post[1]["CpGRatenonsyn23"]>row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn31_post": row_post[1]["CpGRatenonsyn31"],
        "CpGRatenonsyn31_pred": row_pred[1]["CpGRatenonsyn31"],
        "CpGRatenonsyn31_comp": row_post[1]["CpGRatenonsyn31"]>row_pred[1]["CpGRatenonsyn31"],
        "mcmcID": row_post[1]["mcmcID"],
        "geneID": row_post[1]["geneID"],
    }
    k+=1

In [16]:
df_TsCpGRate = pd.DataFrame.from_dict(data=dict_of_stats,orient="index")
df_comp = df_TsCpGRate.groupby(["geneID"]) [[
        "CpGRate_comp",
        "CpGRatesyn_comp",
        "CpGRatenonsyn_comp", 
        "CpGRate12_comp", 
        "CpGRate23_comp", 
        "CpGRate31_comp",

        "CpGRatesyn12_comp", 
        "CpGRatesyn23_comp", 
        "CpGRatesyn31_comp",

        "CpGRatenonsyn12_comp", 
        "CpGRatenonsyn23_comp", 
        "CpGRatenonsyn31_comp",

    ]]\
    .agg([np.mean]).droplevel(level=1,axis=1).reset_index()

In [17]:
df_comp

,geneID,CpGRate_comp,CpGRatesyn_comp,CpGRatenonsyn_comp,CpGRate12_comp,CpGRate23_comp,CpGRate31_comp,CpGRatesyn12_comp,CpGRatesyn23_comp,CpGRatesyn31_comp,CpGRatenonsyn12_comp,CpGRatenonsyn23_comp,CpGRatenonsyn31_comp
0,ACO1,1.000,1.000,1.000,1.000,1.000,1.000,0.0,1.000,1.000,1.000,1.000,1.0
1,ACSL3,1.000,0.998,1.000,1.000,0.822,1.000,0.0,0.902,0.992,1.000,0.136,1.0
2,ADNP,1.000,1.000,1.000,0.248,1.000,0.998,0.0,1.000,0.984,0.248,0.918,1.0
3,AGBL5,1.000,0.992,1.000,0.998,1.000,1.000,0.0,1.000,1.000,0.998,1.000,1.0
4,AMPD1,1.000,1.000,1.000,0.842,1.000,1.000,0.0,1.000,1.000,0.842,1.000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,WDR3,1.000,1.000,1.000,0.680,1.000,1.000,0.0,1.000,1.000,0.680,1.000,1.0
133,WDR36,1.000,1.000,1.000,0.446,1.000,1.000,0.0,1.000,1.000,0.446,1.000,1.0
134,WDR91,1.000,1.000,1.000,0.002,1.000,1.000,0.0,1.000,1.000,0.002,1.000,1.0
135,YTHDF2,0.944,0.970,0.234,0.038,0.564,0.996,0.0,0.426,0.998,0.038,0.804,0.3


In [21]:
df_comp.sort_values(by=["geneID"]).to_csv(f"{ROOT_dir}/reports/map_test_MG-F1x4W.csv",sep="\t")

In [22]:
df_stat = df_comp[[
    "CpGRate_comp",
    "CpGRatesyn_comp",
    "CpGRatenonsyn_comp", 
    
    "CpGRate12_comp", 
    "CpGRate23_comp", 
    "CpGRate31_comp",

    "CpGRatesyn12_comp", 
    "CpGRatesyn23_comp", 
    "CpGRatesyn31_comp",

    "CpGRatenonsyn12_comp", 
    "CpGRatenonsyn23_comp", 
    "CpGRatenonsyn31_comp",
    
    ]].agg([sign_99, sign_95, sign_90 ,"count"])
    
df_stat.round(2).to_csv(ROOT_dir + "/reports/map_test_MG-F1x4W_sign.csv",sep="\t")

In [25]:
df_stat.round(0).T

,sign_99,sign_95,sign_90,count
CpGRate_comp,86.0,88.0,93.0,137.0
CpGRatesyn_comp,80.0,86.0,88.0,137.0
CpGRatenonsyn_comp,89.0,92.0,95.0,137.0
CpGRate12_comp,42.0,45.0,50.0,137.0
CpGRate23_comp,94.0,98.0,98.0,137.0
CpGRate31_comp,91.0,92.0,94.0,137.0
CpGRatesyn12_comp,0.0,0.0,0.0,137.0
CpGRatesyn23_comp,88.0,94.0,99.0,137.0
CpGRatesyn31_comp,81.0,87.0,88.0,137.0
CpGRatenonsyn12_comp,42.0,45.0,50.0,137.0


### Simulation


#### MG-F1x4W

In [ ]:
input_dir = f"{ROOT_dir}/outputs/simulation/pbmpi/MG-F1x4W/"
pattern = "*-M0GTR-*A.TsCpGRate"
df_concat_m0gtr = recover_data_sim(input_dir=input_dir,pattern=pattern, expected_n_lines=1000)
df_concat_m0gtr = df_concat_m0gtr.loc[(df_concat_m0gtr["TpA"]==1)]


In [28]:
df_concat_m0gtr.columns

Index(['mcmcID', 'type', 'CG', 'CG>CA', 'CG>TG', 'CG12', 'CG>CA12', 'CG>TG12',
       'CG23', 'CG>CA23', 'CG>TG23', 'CG31', 'CG>CA31', 'CG>TG31', 'CG>CAsyn',
       'CG>Cnonsyn', 'CG>TGsyn', 'CG>TGnonsyn', 'CG>CAsyn12', 'CG>CAnonsyn12',
       'CG>TGsyn12', 'CG>TGnonsyn12', 'CG>CAsyn23', 'CG>CAnonsyn23',
       'CG>TGsyn23', 'CG>TGnonsyn23', 'CG>CAsyn31', 'CG>CAnonsyn31',
       'CG>TGsyn31', 'CG>TGnonsyn31', 'geneID', 'omega', 'CpG', 'TpA', 'tbl',
       'drawID'],
      dtype='object')

In [ ]:
df_concat_m0gtr["CpGRate"] = (df_concat_m0gtr["CG>TG"]+df_concat_m0gtr["CG>CA"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRatesyn"] = (df_concat_m0gtr["CG>TGsyn"]+df_concat_m0gtr["CG>CAsyn"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRatenonsyn"] = (df_concat_m0gtr["CG>TGnonsyn"]+df_concat_m0gtr["CG>Cnonsyn"])/df_concat_m0gtr["CG"]
df_concat_m0gtr["CpGRate12"] = (df_concat_m0gtr["CG>TG12"]+df_concat_m0gtr["CG>CA12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRate23"] = (df_concat_m0gtr["CG>TG23"]+df_concat_m0gtr["CG>CA23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRate31"] = (df_concat_m0gtr["CG>TG31"]+df_concat_m0gtr["CG>CA31"])/df_concat_m0gtr["CG31"]

df_concat_m0gtr["CpGRatesyn12"] = (df_concat_m0gtr["CG>TGsyn12"]+df_concat_m0gtr["CG>CAsyn12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRatesyn23"] = (df_concat_m0gtr["CG>TGsyn23"]+df_concat_m0gtr["CG>CAsyn23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRatesyn31"] = (df_concat_m0gtr["CG>TGsyn31"]+df_concat_m0gtr["CG>CAsyn31"])/df_concat_m0gtr["CG31"]

df_concat_m0gtr["CpGRatenonsyn12"] = (df_concat_m0gtr["CG>TGnonsyn12"]+df_concat_m0gtr["CG>CAnonsyn12"])/df_concat_m0gtr["CG12"]
df_concat_m0gtr["CpGRatenonsyn23"] = (df_concat_m0gtr["CG>TGnonsyn23"]+df_concat_m0gtr["CG>CAnonsyn23"])/df_concat_m0gtr["CG23"]
df_concat_m0gtr["CpGRatenonsyn31"] = (df_concat_m0gtr["CG>TGnonsyn31"]+df_concat_m0gtr["CG>CAnonsyn31"])/df_concat_m0gtr["CG31"]


df_concat_m0gtr.groupby(["geneID","drawID","omega","CpG","TpA","tbl","type"]).agg([np.mean,np.std])\
    [[
        "CpGRate","CG>TG","CG>CA","CG",\
        "CpGRatesyn","CG>TGsyn","CG>CAsyn",\
        "CpGRatenonsyn","CG>TGnonsyn","CG>Cnonsyn",\

        "CpGRate12","CG>TG12", "CG>CA12", "CG12",\
        "CpGRate23","CG>TG23", "CG>CA23", "CG23",\
        "CpGRate31","CG>TG31", "CG>CA31", "CG31",\
    
        "CpGRatesyn12","CG>TGsyn12", "CG>CAsyn12", \
        "CpGRatesyn23","CG>TGsyn23", "CG>CAsyn23", \
        "CpGRatesyn31","CG>TGsyn31", "CG>CAsyn31", \
    
        "CpGRatenonsyn12","CG>TGnonsyn12", "CG>CAnonsyn12", \
        "CpGRatenonsyn23","CG>TGnonsyn23", "CG>CAnonsyn23", \
        "CpGRatenonsyn31","CG>TGnonsyn31", "CG>CAnonsyn31", \
        
    ]].round(2)

CpGRate         CG>TG          CG>CA  \
                                         mean   std    mean    std    mean   
geneID  drawID omega CpG TpA tbl type                                        
CSRP2BP 0      0.2   1.0 1.0 1.0 post    0.22  0.01  118.33   3.75  112.18   
                                 pred    0.25  0.02  137.60  16.60  112.85   
                     4.0 1.0 1.0 post    0.79  0.02  180.37   4.53  171.64   
                                 pred    0.29  0.03  131.11  16.19  116.94   
                     8.0 1.0 1.0 post    1.17  0.03  196.76   5.72  222.22   
...                                       ...   ...     ...    ...     ...   
WDR91   9      0.2   1.0 1.0 1.0 pred    0.30  0.03  108.76  14.64   91.11   
                     4.0 1.0 1.0 post    0.81  0.03  154.23   3.93  119.68   
                                 pred    0.36  0.04  123.17  17.24   88.42   
                     8.0 1.0 1.0 post    1.37  0.05  148.02   6.35  163.09   
                                 pred    0.38  0.04  115.63  16.32   92.15   

                                                   CG        CpGRatesyn        \
                                         std     mean    std       mean   std   
geneID  drawID omega CpG TpA tbl type                                           
CSRP2BP 0      0.2   1.0 1.0 1.0 post   2.96  1030.30  22.85       0.15  0.01   
                                 pred  13.61   992.40  70.17       0.18  0.02   
                     4.0 1.0 1.0 post   4.21   446.64  11.77       0.55  0.02   
                                 pred  14.63   848.47  72.82       0.21  0.02   
                     8.0 1.0 1.0 post   5.49   359.07   9.98       0.86  0.03   
...                                      ...      ...    ...        ...   ...   
WDR91   9      0.2   1.0 1.0 1.0 pred  12.75   668.89  61.08       0.22  0.03   
                     4.0 1.0 1.0 post   3.07   339.81   9.50       0.54  0.02   
                                 pred  12.88   585.34  58.68       0.26  0.04   
                     8.0 1.0 1.0 post   3.66   227.89   7.37       0.94  0.04   
                                 pred  13.80   552.65  58.29       0.28  0.03   

                                       ... CG>TGnonsyn23       CG>CAnonsyn23  \
                                       ...          mean   std          mean   
geneID  drawID omega CpG TpA tbl type  ...                                     
CSRP2BP 0      0.2   1.0 1.0 1.0 post  ...         15.50  0.80           0.0   
                                 pred  ...         20.28  4.97           0.0   
                     4.0 1.0 1.0 post  ...         39.33  1.20           0.0   
                                 pred  ...         18.92  4.77           0.0   
                     8.0 1.0 1.0 post  ...         48.18  1.37           0.0   
...                                    ...           ...   ...           ...   
WDR91   9      0.2   1.0 1.0 1.0 pred  ...         15.66  3.99           0.0   
                     4.0 1.0 1.0 post  ...         27.79  1.00           0.0   
                                 pred  ...         18.68  5.10           0.0   
                     8.0 1.0 1.0 post  ...         31.45  1.35           0.0   
                                 pred  ...         16.55  4.66           0.0   

                                           CpGRatenonsyn31        \
                                       std            mean   std   
geneID  drawID omega CpG TpA tbl type                              
CSRP2BP 0      0.2   1.0 1.0 1.0 post  0.0            0.26  0.02   
                                 pred  0.0            0.23  0.06   
                     4.0 1.0 1.0 post  0.0            0.53  0.05   
                                 pred  0.0            0.25  0.07   
                     8.0 1.0 1.0 post  0.0            0.41  0.04   
...                                    ...             ...   ...   
WDR91   9      0.2   1.0 1.0 1.0 pred  0.0            0.35  0.10   
         

In [30]:
dict_of_stats = {}
k = 0
rowiter = iter(df_concat_m0gtr.iterrows())
while ((row_post := next(rowiter, None)) is not None):
    try:
        row_pred = next(rowiter)
    except Exception as e:
        print(e,row_post[0])
    dict_of_stats[k] = {
        "CpGRate_post": row_post[1]["CpGRate"],
        "CpGRate_pred": row_pred[1]["CpGRate"],
        "CpGRate_comp": row_post[1]["CpGRate"]>row_pred[1]["CpGRate"],

        "CpGRatesyn_post": row_post[1]["CpGRatesyn"],
        "CpGRatesyn_pred": row_pred[1]["CpGRatesyn"],
        "CpGRatesyn_comp": row_post[1]["CpGRatesyn"]>row_pred[1]["CpGRatesyn"],

        "CpGRatenonsyn_post": row_post[1]["CpGRatenonsyn"],
        "CpGRatenonsyn_pred": row_pred[1]["CpGRatenonsyn"],
        "CpGRatenonsyn_comp": row_post[1]["CpGRatenonsyn"]>row_pred[1]["CpGRatenonsyn"],

        "CpGRate12_post": row_post[1]["CpGRate12"],
        "CpGRate12_pred": row_pred[1]["CpGRate12"],
        "CpGRate12_comp": row_post[1]["CpGRate12"]>row_pred[1]["CpGRate23"],
        "CpGRate23_post": row_post[1]["CpGRate23"],
        "CpGRate23_pred": row_pred[1]["CpGRate23"],
        "CpGRate23_comp": row_post[1]["CpGRate23"]>row_pred[1]["CpGRate23"],
        "CpGRate31_post": row_post[1]["CpGRate31"],
        "CpGRate31_pred": row_pred[1]["CpGRate31"],
        "CpGRate31_comp": row_post[1]["CpGRate31"]>row_pred[1]["CpGRate31"],

        "CpGRatesyn12_post": row_post[1]["CpGRatesyn12"],
        "CpGRatesyn12_pred": row_pred[1]["CpGRatesyn12"],
        "CpGRatesyn12_comp": row_post[1]["CpGRatesyn12"]>row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn23_post": row_post[1]["CpGRatesyn23"],
        "CpGRatesyn23_pred": row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn23_comp": row_post[1]["CpGRatesyn23"]>row_pred[1]["CpGRatesyn23"],
        "CpGRatesyn31_post": row_post[1]["CpGRatesyn31"],
        "CpGRatesyn31_pred": row_pred[1]["CpGRatesyn31"],
        "CpGRatesyn31_comp": row_post[1]["CpGRatesyn31"]>row_pred[1]["CpGRatesyn31"],


        "CpGRatenonsyn12_post": row_post[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn12_pred": row_pred[1]["CpGRatenonsyn12"],
        "CpGRatenonsyn12_comp": row_post[1]["CpGRatenonsyn12"]>row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_post": row_post[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_pred": row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn23_comp": row_post[1]["CpGRatenonsyn23"]>row_pred[1]["CpGRatenonsyn23"],
        "CpGRatenonsyn31_post": row_post[1]["CpGRatenonsyn31"],
        "CpGRatenonsyn31_pred": row_pred[1]["CpGRatenonsyn31"],
        "CpGRatenonsyn31_comp": row_post[1]["CpGRatenonsyn31"]>row_pred[1]["CpGRatenonsyn31"],

        "mcmcID": row_post[1]["mcmcID"],
        "geneID": row_post[1]["geneID"],
        "omega" : row_post[1]["omega"],
        "CpG" : row_post[1]["CpG"],
        "TpA" : row_post[1]["TpA"],
        "tbl" : row_post[1]["tbl"],
        "drawID": row_post[1]["drawID"], 
    }
    k+=1

In [31]:
df_comp = pd.DataFrame.from_dict(data=dict_of_stats,orient="index")\
    .groupby(["geneID","mcmcID","drawID","omega","CpG","TpA","tbl"])\
    [[
        "CpGRate_comp",
        "CpGRatesyn_comp",
        "CpGRatenonsyn_comp", 
        "CpGRate12_comp", 
        "CpGRate23_comp", 
        "CpGRate31_comp",

        "CpGRatesyn12_comp", 
        "CpGRatesyn23_comp", 
        "CpGRatesyn31_comp",

        "CpGRatenonsyn12_comp", 
        "CpGRatenonsyn23_comp", 
        "CpGRatenonsyn31_comp",

    ]]\
        .agg([np.mean]).droplevel(level=1,axis=1).reset_index()

In [32]:
df_comp

,geneID,mcmcID,drawID,omega,CpG,TpA,tbl,CpGRate_comp,CpGRatesyn_comp,CpGRatenonsyn_comp,CpGRate12_comp,CpGRate23_comp,CpGRate31_comp,CpGRatesyn12_comp,CpGRatesyn23_comp,CpGRatesyn31_comp,CpGRatenonsyn12_comp,CpGRatenonsyn23_comp,CpGRatenonsyn31_comp
0,CSRP2BP,100,0,0.2,1.0,1.0,1.0,0.1,0.0,0.6,0.0,0.5,0.4,0.0,0.8,0.1,1.0,0.3,0.7
1,CSRP2BP,100,0,0.2,4.0,1.0,1.0,1.0,1.0,1.0,0.4,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
2,CSRP2BP,100,0,0.2,8.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
3,CSRP2BP,100,1,0.2,1.0,1.0,1.0,0.5,0.5,0.8,0.0,0.8,0.1,0.0,0.9,0.1,1.0,0.3,0.8
4,CSRP2BP,100,1,0.2,4.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,WDR91,198,8,0.2,4.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
14996,WDR91,198,8,0.2,8.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
14997,WDR91,198,9,0.2,1.0,1.0,1.0,0.7,0.6,0.9,0.0,0.7,0.1,0.0,0.8,0.2,1.0,0.1,0.4
14998,WDR91,198,9,0.2,4.0,1.0,1.0,1.0,1.0,1.0,0.5,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0


In [33]:
df_comp.sort_values(by=["geneID","omega","CpG","TpA","tbl"]).to_csv(ROOT_dir + "/reports/map_test_m0gtr_m0gtr.csv",sep="\t")

In [34]:
df_comp.sort_values(by=["geneID","omega","CpG","TpA","tbl"])

,geneID,mcmcID,drawID,omega,CpG,TpA,tbl,CpGRate_comp,CpGRatesyn_comp,CpGRatenonsyn_comp,CpGRate12_comp,CpGRate23_comp,CpGRate31_comp,CpGRatesyn12_comp,CpGRatesyn23_comp,CpGRatesyn31_comp,CpGRatenonsyn12_comp,CpGRatenonsyn23_comp,CpGRatenonsyn31_comp
0,CSRP2BP,100,0,0.2,1.0,1.0,1.0,0.1,0.0,0.6,0.0,0.5,0.4,0.0,0.8,0.1,1.0,0.3,0.7
3,CSRP2BP,100,1,0.2,1.0,1.0,1.0,0.5,0.5,0.8,0.0,0.8,0.1,0.0,0.9,0.1,1.0,0.3,0.8
6,CSRP2BP,100,2,0.2,1.0,1.0,1.0,0.3,0.4,0.4,0.0,0.4,0.6,0.0,0.4,0.6,1.0,0.4,0.2
9,CSRP2BP,100,3,0.2,1.0,1.0,1.0,0.0,0.0,0.2,0.0,0.0,0.3,0.0,0.0,0.3,1.0,0.1,0.3
12,CSRP2BP,100,4,0.2,1.0,1.0,1.0,0.5,0.5,0.6,0.0,0.6,0.1,0.0,0.5,0.1,1.0,0.7,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14987,WDR91,198,5,0.2,8.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
14990,WDR91,198,6,0.2,8.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
14993,WDR91,198,7,0.2,8.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
14996,WDR91,198,8,0.2,8.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0


In [ ]:
df_comp.groupby(by=["CpG","TpA","omega","tbl"])[[
    "CpGRate_comp",
    "CpGRatesyn_comp",
    "CpGRatenonsyn_comp", 
    
    "CpGRate12_comp", 
    "CpGRate23_comp", 
    "CpGRate31_comp",

    "CpGRatesyn12_comp", 
    "CpGRatesyn23_comp", 
    "CpGRatesyn31_comp",

    "CpGRatenonsyn12_comp", 
    "CpGRatenonsyn23_comp", 
    "CpGRatenonsyn31_comp",
    
    ]].agg([sign_99, sign_95, sign_90 ,"count"])

CpGRate_comp                       CpGRatesyn_comp          \
                       sign_99 sign_95 sign_90 count         sign_99 sign_95   
CpG TpA omega tbl                                                              
1.0 1.0 0.2   1.0          9.1     9.1   18.58  5000            9.66    9.66   
4.0 1.0 0.2   1.0        100.0   100.0  100.00  5000          100.00  100.00   
8.0 1.0 0.2   1.0        100.0   100.0  100.00  5000          100.00  100.00   

                                CpGRatenonsyn_comp          ...  \
                  sign_90 count            sign_99 sign_95  ...   
CpG TpA omega tbl                                           ...   
1.0 1.0 0.2   1.0    18.3  5000                9.4     9.4  ...   
4.0 1.0 0.2   1.0   100.0  5000              100.0   100.0  ...   
8.0 1.0 0.2   1.0   100.0  5000              100.0   100.0  ...   

                  CpGRatenonsyn12_comp       CpGRatenonsyn23_comp          \
                               sign_90 count              sign_99 sign_95   
CpG TpA omega tbl                                                           
1.0 1.0 0.2   1.0                78.38  5000                 7.88    7.88   
4.0 1.0 0.2   1.0               100.00  5000                98.76   98.76   
8.0 1.0 0.2   1.0               100.00  5000                99.96   99.96   

                                CpGRatenonsyn31_comp                        
                  sign_90 count              sign_99 sign_95 sign_90 count  
CpG TpA omega tbl                                                           
1.0 1.0 0.2   1.0   15.78  5000                  9.7     9.7   20.54  5000  
4.0 1.0 0.2   1.0   99.42  5000                 76.8    76.8   89.20  5000  
8.0 1.0 0.2   1.0  100.00  5000                 93.4    93.4   97.54  5000  

[3 rows x 48 columns]